#  ```insert_subgraph_with_mappings```
This example demonstrates how to use the ```insert_subgraph_between_tensors API``` to optimize an ONNX model by replacing a ```Sub``` node with an equivalent sequence of ```Mul``` and ```Add``` nodes. This targeted graph surgery enables efficient model modifications without needing to rebuild or retrain the entire model.<br><b>This method requires PyTorch to be installed, as it uses PyTorch to generate the replacement ONNX subgraph.</b>

## <b>Setup</b>

In [ ]:
%pip install torch onnx onnx-graphsurgeon numpy onnxruntime torch netron

In [1]:
# Step 1: Setup and Imports
import onnx
import onnx_graphsurgeon as gs
import numpy as np
import logging

import tempfile
import os
import torch
import torch.nn as nn

# Optional: Set logging level for easier debugging
logging.basicConfig(level=logging.INFO)

## <b>Problem</b>
In this example, we’ll show how you can use the ````insert_subgraph_between_tensors```` API to swap out a ```Sub``` node in your ONNX graph for a mathematically equivalent sequence: first multiply one input by -1 (using ```Mul```), then add it to the other input (using ```Add```).

## <b>Code Flow</b>
Below is a step-by-step example showing how you can use this API to customize your ONNX graphs for your own needs. With just a basic understanding of ONNX GraphSurgeon/ Pytorch, you can easily perform targeted graph surgery and optimize your models, no need to retrain or rebuild from scratch!

### <b>Step 0 : Create/Load the model</b>
Load or create your model which needs to be modified

In [6]:
model = onnx.load("./example_models/sim_test_multi_sub.onnx") # <- Replace with your ONNX model path
graph = gs.import_onnx(model)

import netron
# Launch Netron to visualize the ONNX model in your browser
netron.start("./example_models/sim_test_multi_sub.onnx")


INFO:netron.server:Serving './example_models/sim_test_multi_sub.onnx' at http://localhost:15376


('localhost', 15376)

### <b>Step 1 : Load the API</b>
Load the API from ```common.py```

In [7]:
from osrt_model_tools.onnx_tools.tidl_onnx_model_optimizer.src.common import insert_subgraph_with_mappings

### <b>Step 2(a) : Use PyTorch to Create the Replacement Subgraph (Suggested) </b>
You can leverage PyTorch to define the new subgraph. The following method demonstrates how to use PyTorch to construct the replacement subgraph, highlighting important considerations and best practices for this approach.

#### <b> Step 2.1(a) : Define a custom Transformation Function </b>

In [9]:
def sub_to_muladd_pytorch(graph: gs.Graph):
    """
    Replace all Sub nodes with an equivalent Mul+Add subgraph using PyTorch for subgraph generation.
    """
    nodes = graph.nodes
    idx = 0
    for node in nodes:
        if node.op == "Sub" and len(node.inputs) == 2:
            input1 = node.inputs[0]
            input2 = node.inputs[1]
            output = node.outputs[0]
            input1_name = input1.name
            input2_name = input2.name
            output_name = output.name
            input1_shape = input1.shape
            input2_shape = input2.shape
            output_shape = output.shape
            suffix = f"_sub2muladd_{idx}"

            # 1. Define the PyTorch module for Sub as Mul+Add
            class SubToMulAddModule(nn.Module):
                def forward(self, a, b):
                    return a + (b * -1) 

            # 2. Export the module to ONNX
            dummy_a = torch.randn(*input1_shape)
            dummy_b = torch.randn(*input2_shape)
            with tempfile.NamedTemporaryFile(suffix=".onnx", delete=False) as tmpfile:
                torch.onnx.export(
                    SubToMulAddModule(),
                    (dummy_a, dummy_b),
                    tmpfile.name,
                    input_names=["input1", "input2"],
                    output_names=["output"],
                    opset_version= graph.opset
                )
                muladd_model = onnx.load(tmpfile.name)
                muladd_model = onnx.shape_inference.infer_shapes(muladd_model)      # GraphSurgeon requires shapes to be inferred for the input and output tensors
            os.remove(tmpfile.name)

            # 3. Import the ONNX subgraph into GraphSurgeon
            muladd_gs = gs.import_onnx(muladd_model)

            # 4. Set the input and output names in the subgraph
            input_mapping = {muladd_gs.inputs[0].name: input1_name, muladd_gs.inputs[1].name: input2_name}
            output_mapping = {muladd_gs.outputs[0].name: output_name}

            # 5. Use the API to insert the new subgraph in place of the Sub node
            success = insert_subgraph_with_mappings(
                graph,
                input_mapping,
                output_mapping,
                muladd_gs,
                suffix          # This step is optional, but helps to avoid name conflicts, manually handle naming if not passed
            )
            if success:
                print(f"Sub→Mul+Add replacement succeeded for node {node.name}.")
            else:
                print(f"Sub→Mul+Add replacement failed for node {node.name}.")
            idx += 1

#### <b>Step 2.2(a) : Call the function</b>
Pass the original graph to the function and save the transformed graph, if required

In [10]:
sub_to_muladd_pytorch(graph)
onnx.save(gs.export_onnx(graph), "./example_models/optim_sim_test_multi_sub.onnx")
netron.start("./example_models/optim_sim_test_multi_sub.onnx")

INFO:netron.server:Serving './example_models/optim_sim_test_multi_sub.onnx' at http://localhost:21382


Sub→Mul+Add replacement succeeded for node sub1.
Sub→Mul+Add replacement succeeded for node sub2.
Sub→Mul+Add replacement succeeded for node sub3.


('localhost', 21382)

### <b>Step 2(b) : Use GraphSurgeon to Create the Replacement Subgraph (Advanced Users) </b>
You can leverage GraphSurgeon to define the new subgraph. The following method demonstrates how to use GraphSurgeon to construct the replacement subgraph, highlighting important considerations and best practices for this approach. This 

#### <b>Step 2.1(b) : Define a class</b> (Optional)
Defining a class for your replacement pattern is a good practice. It helps keep your code modular and organized, making it easier to visualize and manage all the important details needed for optimization. While this step is optional, it’s highly recommended for clarity and maintainability, especially as your graph surgery tasks become more complex.

In [11]:
# Replacement Pattern
class SubPattern:
    def __init__(self):
        # Required for creating the new subgraph
        self.sub_node_idx = -1
        self.idx = -1
        self.input1 = -1
        self.input2 = -1
        self.input1_shape = None
        self.input2_shape = None
        self.output = -1
        self.output_shape = None
        # to check if replacement is ready
        self.replace_ready = False
        # Required for the API
        self.new_subgraph = None
        self.input_mapping = None
        self.output_mapping = None
        
    def optimize(self, graph: gs.Graph):
        """
        Build a new subgraph (Mul + Add) using ONNX GraphSurgeon, matching the original Sub node's features.
        """
        # Ensure unique name for every node !IMP
        suffix = f"_sub_id_{self.idx}"

        # Use extracted names and shapes
        # shapes necessary for input and output variables
        input1_name = self.input1    # <- can create custom names for new subgraph if needed, make sure to map accordingly
        input2_name = self.input2 
        output_name = self.output 
        input1_shape = self.input1_shape
        input2_shape = self.input2_shape
        output_shape = self.output_shape

        # Create input variables (matching the original graph)
        in1 = gs.Variable(input1_name, dtype=np.float32, shape=input1_shape)
        in2 = gs.Variable(input2_name, dtype=np.float32, shape=input2_shape)

        # Mul node: in2 * -1
        neg_const = gs.Constant(f"{input2_name}_neg1{suffix}", values=np.array([-1], dtype=np.float32))
        mul_out = gs.Variable(f"{input2_name}_neg{suffix}", dtype=np.float32, shape=input2_shape)
        mul_node = gs.Node(op="Mul", name=f"Mul{suffix}", inputs=[in2, neg_const], outputs=[mul_out])

        # Add node: in1 + (in2 * -1)
        add_out = gs.Variable(output_name, dtype=np.float32, shape=output_shape)
        add_node = gs.Node(op="Add", name=f"Add{suffix}", inputs=[in1, mul_out], outputs=[add_out])

        # Build the subgraph
        self.new_subgraph = gs.Graph(
            nodes=[mul_node, add_node],
            inputs=[in1, in2],
            outputs=[add_out]
        )

        # Set up mappings for the API
        # The mapping is like followes : {new_subgraph_input_tensor_name: original_graph_input_tensor_name}
        self.input_mapping = {
            input1_name: input1_name,
            input2_name: input2_name,
        }
        self.output_mapping = {
            output_name: output_name
        }
        self.replace_ready = True

<span style="font-size:10px"><ul><li><i>Attributes:</i></li>
<ul>
<li> <code>sub_node_idx</code>, <code>idx</code>: Indices to help identify and track the node in the graph.</li>
<li><code>input1</code>, <code>input2</code>, <code>output</code>: Names of the input and output tensors for the original <code>Sub</code> node.</li>
<li><code>input1_shape</code>, <code>input2_shape</code>, <code>output_shape</code>: Shapes of the input and output tensors.</li>
<li><code>replace_ready</code>: Flag indicating if the replacement subgraph is ready.</li>
<li><code>new_subgraph</code>: The ONNX GraphSurgeon subgraph that will replace the original Sub node.</li>
<li><code>input_mapping</code>, <code>output_mapping</code>: Dictionaries mapping the new subgraph’s inputs/outputs to the original graph’s tensors.</li>
</ul>

<li><i>optimize Method</i></li>
This method builds a new subgraph that performs the same operation as<code>A - B</code> by computing <code>A + (B * -1)</code>. It:
<ul>
<li>Creates new input variables that match the original node’s inputs.</li>
<li>Adds a <code>Mul</code> node to multiply the second input by -1.</li>
<li>Adds an <code>Add</code> node to add the first input and the negated second input.</li>
<li>Packages these nodes into a new subgraph, with proper input/output mappings for easy integration into the original graph.</li>
</ul>


</ul>
</span>


#### <b>Step 2.2(b) : Define a Transformation Function</b>
Define a function which stores the various attributes of ```Sub``` Node in the ```SubPattern``` Class

In [ ]:
def tidl_sub_to_matmuladd(graph: gs.Graph):
    """
    Replace all Sub nodes in the graph with an equivalent Mul+Add subgraph.
    This demonstrates how to use the API for targeted ONNX graph surgery.
    """
    nodes = graph.nodes
    num_subs = 0  # Counter for found Sub nodes
    sub_nodes = []  # Store all SubPattern instances for reference

    for idx, node in enumerate(nodes):
        # Prepare a pattern object for each node
        sub = SubPattern()
        sub.sub_node_idx = idx
        sub.idx = num_subs

        # Check if the node is a Sub op with two variable inputs
        if (
            node.op == "Sub"
            and len(node.inputs) == 2
            and isinstance(node.inputs[0], gs.Variable)
            and isinstance(node.inputs[1], gs.Variable)
        ):
            # Extract input/output names and shapes for the pattern
            sub.input1 = node.inputs[0].name
            sub.input2 = node.inputs[1].name
            sub.output = node.outputs[0].name
            sub.input1_shape = node.inputs[0].shape
            sub.input2_shape = node.inputs[1].shape
            sub.output_shape = node.outputs[0].shape
            num_subs += 1

            logging.info(
                f"Found subgraph pattern at index {idx}: {sub.input1} - {sub.input2} -> {sub.output}"
            )

            # Build the replacement Mul+Add subgraph
            sub.optimize(graph)
            sub_nodes.append(sub)
            
            print(sub.input_mapping)
            print(sub.output_mapping)

            # Use the API to insert the new subgraph in place of the Sub node
            success = insert_subgraph_with_mappings(
                graph,
                sub.input_mapping,
                sub.output_mapping,
                sub.new_subgraph,
            )

            if success:
                print("Subgraph replacement succeeded.")
                # Optionally save the intermediate graph here
                # onnx.save(gs.export_onnx(graph), "<path>")
            else:
                print(
                    "Subgraph replacement failed due to external dependencies or shape mismatch."
                )

    # Save the final optimized graph
    onnx.save(gs.export_onnx(graph), "./example_models/optim_test_multi_sub.onnx")
    netron.start("./example_models/optim_test_multi_sub.onnx")
    
# Call the function    
tidl_sub_to_matmuladd(graph)